This script takes a pre-optimized ensemble configuration (from a JSON file)
and applies it to a hold-out test set to get final performance metrics.
It is the final step after running the Optuna weight optimization script.

This version has been refactored to include bootstrap resampling to calculate
95% confidence intervals for all relevant performance metrics.

1. Set Up Environment

In [ ]:
# --- 1. Imports ---
print("Importing libraries...")
# (Same imports as your original script)
!pip install albumentations torchinfo
!pip install git+https://github.com/qubvel/segmentation_models.pytorch

In [ ]:
!apt-get update -qq
!apt-get install -y -qq \
    texlive-latex-base \
    texlive-latex-recommended \
    texlive-latex-extra \
    texlive-fonts-recommended \
    texlive-fonts-extra

In [ ]:
!pdflatex --version

# --- 1. Imports ---

In [ ]:
# --- 1. Imports ---
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image
import gc
import time
from tqdm import tqdm
import torchvision.models as models
from datetime import datetime
import torch.nn.functional as F
from google.colab import userdata
import random
import seaborn as sns
import shutil
import smtplib
from email.mime.text import MIMEText
from torch.cuda.amp import autocast
import cv2
import segmentation_models_pytorch as smp
import timm
from torchinfo import summary
import pandas as pd
import json
from sklearn.metrics import auc as sklearn_auc
import warnings
import albumentations as A
from albumentations.pytorch import ToTensorV2
import zipfile
import re
from collections import defaultdict # <-- Make sure this is imported
import subprocess
import sys

In [ ]:
# --- 5. Configuration & Setup ---
print("Configuring environment...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ### REFACTORED: Simplified Configuration ###
# This is the only path you need to set. Point it to the output of the Optuna script.
ENSEMBLE_META_PATH = "RESULTS_REPORT_ENSEMBLE/DIAGSET/REINHARD/ENSEMBLE_OPTIMIZATION_17_12_2025_02_32_32.json" # <--- UPDATE THIS PATH

BATCH_SIZE = 32
WORKERS = 2
SEED = 42

# --- Dataset Source Path (for the TEST set) ---
DATASET_ZIP_DIR = 'IA_MEDICA_SAMPLES/DIAGSET/REINHARD'
base_data_dir = '/content/dataset'

DECODER_DROPOUT = 0

WHERE_WAS_CREATED = '/content/drive/MyDrive/Personal_Drive_Bruno/'
CURRENT_ENV = '/content/drive/MyDrive/'

# --- NEW: Ensemble Method Configuration ---
ENSEMBLE_METHOD = "WeightedAverage" # Options: "WeightedAverage", "MajorityVote"

VOTING_THRESHOLD = 'optimize'

STATS_SAMPLE_SIZE = None
MEAN = [0.5631743144356475, 0.3779451521216607, 0.6975475908629748]
STD = [0.07002276986060542, 0.05347828888148516, 0.047768733057653855]

In [ ]:
def change_path(path, possible_paths=None):
    if possible_paths is None:
        possible_paths = ['/content/drive/MyDrive/Personal_Drive_Bruno/','/content/drive/MyDrive/']
    for p in possible_paths:
      path=os.path.join(p,path)
      if os.path.exists(path):
        print(f"Path changed succesfully: {path}")
        return path
    raise ValueError(f"The path {path} does not match any known environments.")

In [ ]:
def change_paths(path):
  if not os.path.exists(path):
    new_path = str(path).replace(WHERE_WAS_CREATED, CURRENT_ENV)
    if os.path.exists(new_path):
      path = new_path
      print(f"Updated path to {path}")
    else:
      print(f"Path not found: {new_path}")
  return path

In [ ]:
ENSEMBLE_META_PATH = change_path(ENSEMBLE_META_PATH)
DATASET_ZIP_DIR = change_path(DATASET_ZIP_DIR)

In [ ]:
# --- Paths to the HOLD-OUT TEST SET ---
train_cancer_image_dir = os.path.join(base_data_dir, 'TRAIN/CANCER')
train_cancer_mask_dir = os.path.join(base_data_dir, 'TRAIN/CANCER_MASK')
train_not_cancer_image_dir = os.path.join(base_data_dir, 'TRAIN/NOT_CANCER')
train_not_cancer_mask_dir = os.path.join(base_data_dir, 'TRAIN/NOT_CANCER_MASK')
test_cancer_image_dir = os.path.join(base_data_dir, 'TEST/CANCER')
test_cancer_mask_dir = os.path.join(base_data_dir, 'TEST/CANCER_MASK')
test_not_cancer_image_dir = os.path.join(base_data_dir, 'TEST/NOT_CANCER')
test_not_cancer_mask_dir = os.path.join(base_data_dir, 'TEST/NOT_CANCER_MASK')

In [ ]:
# --- Seeding ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Configuration complete.")

In [ ]:
# --- Helper Functions ---
def get_formatted_datetime_string():
  now = datetime.now()
  return now.strftime("%d_%m_%Y_%H_%M_%S")

In [ ]:
def get_model(architecture, encoder,validation=False):

  if validation:
    aux_params=None
  else:
    aux_params=dict(dropout=DECODER_DROPOUT, classes=2)

  encoder_weights = None if validation else "imagenet"

  if architecture=="SWIN":
    model = smp.Unet(
    encoder_name=encoder,
    encoder_weights=encoder_weights,
    in_channels=3,
    classes=2,
    activation=None,
    decoder_attention_type=None,
    aux_params=aux_params)
  elif architecture=="DEEPLABV3PLUS":
    model = smp.DeepLabV3Plus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="INCEPTIONRESNETV2":
    model = smp.Unet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="DPT":
    model = smp.DPT(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        decoder_readout='ignore',
        aux_params=aux_params)
  elif architecture=="UNET++":
    model = smp.UnetPlusPlus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="FPN":
    model = smp.FPN(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="SEGFORMER":
    model = smp.Segformer(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="MANET":
    model = smp.MAnet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="UPERNET":
    model = smp.UPerNet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  else:
    raise ValueError(f"Unknown architecture: {architecture}")

  return model

In [ ]:
def predict_with_tta(model, images):
    """
    Run the model with simple test-time augmentations and average
    the cancer-channel probabilities.

    Args:
        model: segmentation model (may return (logits, aux))
        images: tensor [B, 3, H, W] already on the correct device

    Returns:
        probs_cancer: [B, H, W] averaged over TTA transforms
                      (foreground / cancer channel)
    """
    tta_probs = []

    def forward_pass(x):
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            out = model(x)
            if isinstance(out, (tuple, list)):
                out = out[0]

            # Binary (1-channel) vs multi-class (2-channel) outputs
            if out.shape[1] == 1:
                # out: [B, 1, H, W] → sigmoid, then squeeze channel
                probs = torch.sigmoid(out).squeeze(1)  # [B, H, W]
            else:
                # out: [B, 2, H, W] → softmax cancer channel
                probs = torch.softmax(out, dim=1)[:, 1, :, :]  # [B, H, W]
        return probs

    # 1) Original
    probs = forward_pass(images)
    tta_probs.append(probs)

    # 2) Horizontal flip
    imgs_h = torch.flip(images, dims=[3])          # flip width
    probs_h = forward_pass(imgs_h)
    probs_h = torch.flip(probs_h, dims=[2])        # unflip prediction (width axis)
    tta_probs.append(probs_h)

    # 3) Vertical flip
    imgs_v = torch.flip(images, dims=[2])          # flip height
    probs_v = forward_pass(imgs_v)
    probs_v = torch.flip(probs_v, dims=[1])        # unflip prediction (height axis)
    tta_probs.append(probs_v)

    # Stack: [T, B, H, W] → mean over TTA transforms → [B, H, W]
    probs_cancer = torch.stack(tta_probs, dim=0).mean(dim=0)
    return probs_cancer

4. Create Custom Dataset

In [ ]:
print("Defining simplified ProstateCancerDataset...")
class ProstateCancerDataset(Dataset):
    def __init__(
        self,
        cancer_image_dir,
        cancer_mask_dir,
        not_cancer_image_dir,
        not_cancer_mask_dir,
        mean=None,
        std=None,
        compute_stats=False,
        stats_sample_size=None,  # e.g. 10_000 patches to speed up estimation
    ):
        self.cancer_image_dir = cancer_image_dir
        self.cancer_mask_dir = cancer_mask_dir
        self.not_cancer_image_dir = not_cancer_image_dir
        self.not_cancer_mask_dir = not_cancer_mask_dir

        self.patient_ids = []

        # Combine and store file paths and labels
        self.image_paths = []
        self.mask_paths = []
        self.labels = []

        # Compile the regex once for efficiency
        patient_id_pattern = re.compile(r'PATIENT_(\d+)_')

        # --- Process CANCER images ---
        cancer_images = []
        if os.path.isdir(self.cancer_image_dir):
            cancer_images = [f for f in os.listdir(cancer_image_dir) if f.lower().endswith('.png')]

        for img_name in cancer_images:
            mask_path = os.path.join(self.cancer_mask_dir, img_name)
            match = patient_id_pattern.search(img_name)
            if os.path.isfile(mask_path) and match:
                self.image_paths.append(os.path.join(self.cancer_image_dir, img_name))
                self.mask_paths.append(mask_path)
                self.labels.append(1)
                self.patient_ids.append(match.group(1))

        # --- Process NOT_CANCER images ---
        not_cancer_images = []
        if os.path.isdir(self.not_cancer_image_dir):
            not_cancer_images = [f for f in os.listdir(not_cancer_image_dir) if f.lower().endswith('.png')]

        for img_name in not_cancer_images:
            mask_path = os.path.join(self.not_cancer_mask_dir, img_name)
            match = patient_id_pattern.search(img_name)
            if os.path.isfile(mask_path) and match:
                self.image_paths.append(os.path.join(self.not_cancer_image_dir, img_name))
                self.mask_paths.append(mask_path)
                self.labels.append(0)
                self.patient_ids.append(match.group(1))

        # ------------------------------------------------------------------
        #  NEW: dataset-specific mean/std (after stain normalization)
        # ------------------------------------------------------------------
        if mean is not None and std is not None:
            # Use externally provided stats (recommended for VAL / TEST)
            self.mean = mean if isinstance(mean, list) else list(mean)
            self.std = std if isinstance(std, list) else list(std)
            print(f"[ProstateCancerDataset] Using provided mean/std: "
                  f"mean={self.mean}, std={self.std}")
        elif compute_stats:
            # Compute stats from this dataset (recommended for TRAIN set)
            print("[ProstateCancerDataset] Computing dataset-specific mean/std...")
            self.mean, self.std = self._compute_dataset_mean_std(
                max_samples=stats_sample_size
            )
            print(f"[ProstateCancerDataset] Computed mean: {self.mean}")
            print(f"[ProstateCancerDataset] Computed std:  {self.std}")
        else:
            # Fallback: ImageNet stats (old behavior, but less ideal scientifically)
            self.mean = [0.485, 0.456, 0.406]
            self.std = [0.229, 0.224, 0.225]
            print("[ProstateCancerDataset] Using ImageNet mean/std "
                  "(no dataset-specific stats requested).")

        # --- Base Transformation (Applied to ALL data) ---
        # NOTE: Albumentations.Normalize expects mean/std in [0,1] scale when
        # max_pixel_value=255.0 (default). We computed them in that scale.
        self.base_transform = A.Compose([
            A.Resize(224, 224, interpolation=cv2.INTER_LINEAR),  # Specify interpolation
            A.Normalize(mean=self.mean, std=self.std),
            ToTensorV2(),  # Handles image scaling & channel order
        ])

    def _compute_dataset_mean_std(self, max_samples=None):
        """
        Compute per-channel mean and std over this dataset's images.

        Args:
            max_samples (int or None): if set, randomly sample up to this many
                images to estimate stats (for speed). If None, use all images.

        Returns:
            mean (list of 3 floats), std (list of 3 floats) in [0,1] scale.
        """
        # If dataset is empty, fall back to ImageNet to avoid crashes
        if len(self.image_paths) == 0:
            print("[ProstateCancerDataset] WARNING: No images found; "
                  "falling back to ImageNet stats.")
            return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

        # Decide which indices to use
        indices = np.arange(len(self.image_paths))
        if max_samples is not None and max_samples < len(indices):
            np.random.shuffle(indices)
            indices = indices[:max_samples]

        n_pixels_total = 0
        channel_sum = np.zeros(3, dtype=np.float64)
        channel_sum_sq = np.zeros(3, dtype=np.float64)

        for idx in indices:
            img_path = self.image_paths[idx]
            img = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if img is None:
                continue  # skip broken images

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img.astype(np.float32) / 255.0  # scale to [0,1]

            # Flatten to (N, 3)
            img_flat = img.reshape(-1, 3)

            n_pixels = img_flat.shape[0]
            n_pixels_total += n_pixels

            channel_sum += img_flat.sum(axis=0)
            channel_sum_sq += (img_flat ** 2).sum(axis=0)

        if n_pixels_total == 0:
            print("[ProstateCancerDataset] WARNING: Failed to load any pixels "
                  "while computing stats; falling back to ImageNet stats.")
            return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

        mean = channel_sum / n_pixels_total
        var = (channel_sum_sq / n_pixels_total) - mean ** 2
        std = np.sqrt(np.maximum(var, 1e-12))

        return mean.tolist(), std.tolist()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        patient_id = self.patient_ids[idx]

        try:
            image = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if image is None: raise IOError("cv2.imread failed for image")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert to RGB for consistency if needed downstream

            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None: raise IOError("cv2.imread failed for mask")

        except Exception as e:
            print(f"Error loading image/mask: {img_path} / {mask_path} - {e}")
            # Return None tuple, handled by collate_fn
            return None, None, None

        # Create Two-Channel Mask (One-Hot Encode) before transform
        mask = mask.astype(np.uint8)
        two_channel_mask = np.zeros((mask.shape[0], mask.shape[1], 2), dtype=np.float32)
        two_channel_mask[mask == 0, 0] = 1.0  # Background channel
        two_channel_mask[mask != 0, 1] = 1.0  # Cancer channel

        # Apply the base transformations
        try:
            # Pass mask correctly (shape H, W, C)
            augmented = self.base_transform(image=image, mask=two_channel_mask)
            final_image = augmented['image'] # Shape (C, H, W), FloatTensor, Normalized
            final_mask = augmented['mask']   # Shape (C, H, W), FloatTensor, Values 0.0 or 1.0

            # Ensure mask shape is (2, 224, 224)
            if final_mask.shape[0] != 2:
                 resized_mask = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_NEAREST)
                 two_channel_mask_resized = np.zeros((224, 224, 2), dtype=np.float32)
                 two_channel_mask_resized[resized_mask == 0, 0] = 1.0
                 two_channel_mask_resized[resized_mask != 0, 1] = 1.0
                 final_mask = torch.from_numpy(two_channel_mask_resized).permute(2, 0, 1) # HWC -> CHW

            # Final check on mask shape
            if final_mask.shape != (2, 224, 224):
                 raise ValueError(f"Final mask shape is incorrect: {final_mask.shape}")


        except Exception as e:
             print(f"Error applying transform to {os.path.basename(img_path)}: {e}")
             return None, None, None


        return final_image, final_mask, patient_id

    def get_class_counts(self):
        """Returns the counts of cancer (1) and not-cancer (0) samples."""
        counts = np.bincount(self.labels)
        not_cancer_count = counts[0] if len(counts) > 0 else 0
        cancer_count = counts[1] if len(counts) > 1 else 0
        return {'CANCER': cancer_count, 'NOT_CANCER': not_cancer_count}

print("Dataset definition complete.")

In [ ]:
def clear_gpu():
    print("Clearing GPU cache...")
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception as e:
            # Optional: only print if something really went wrong
            print(f"[clear_gpu] Warning: {e}")
    print("GPU cache cleared.")

In [ ]:
# Helper function for correct metric calculation
def calculate_metrics(tp, fp, fn, tn):
    """
    Calculates single-class performance metrics from confusion matrix counts
    for the *tumor* (positive) class.

    - Dice and IoU are defined only for cases with at least one positive pixel
      in the ground truth (tp + fn > 0). For purely negative cases, Dice/IoU
      are returned as np.nan and should be excluded from tumor-only averages.
    """
    # Check for a true negative case (no positive pixels in ground truth or prediction)
    # A more precise check is for no positive pixels in the ground truth
    is_positive_case = (tp + fn) > 0

    if is_positive_case:
        dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    else:
        # For cases with no cancer in the ground truth, Dice and IoU are not applicable.
        # We return NaN, so they can be excluded from macro-averages.
        dice = np.nan
        iou = np.nan

    return {
        'dice': dice,
        'iou': iou,
        'accuracy': (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0,
        'tpr': tp / (tp + fn) if (tp + fn) > 0 else 0.0, # Sensitivity
        'tnr': tn / (tn + fp) if (tn + fp) > 0 else 1.0, # Specificity (is 1.0 for a perfect negative)
        'precision': tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        'fnr': fn / (fn + tp) if (fn + tp) > 0 else 0.0
    }

In [ ]:
# Helper for printing
def print_metric_line(metric, pe, ci):
    ci_str = f"(95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])" if not np.isnan(ci[0]) else "(CI not calculated; n_patients < 20)"
    print(f"  {metric.replace('_',' ').title():<12}: {pe:.4f}  {ci_str}")

In [ ]:
def safe_tensor_weights(model_weights, device, n_models):
    if model_weights is None:
        w = torch.ones(n_models, device=device, dtype=torch.float32) / float(n_models)
    else:
        w = torch.tensor(model_weights, device=device, dtype=torch.float32)
        w = w / (w.sum() + 1e-12)
    return w

In [ ]:
def analyze_ensemble_metrics(models_list, constituent_models_info, model_weights,
                            test_loader, device, optimal_threshold, train_mean, train_std, num_auc_steps=101):
    """
    Calculates performance metrics for the ensemble.

    ### FINAL & COMPLETE VERSION (with Patient-Level Bootstrap) ###
    This function calculates and reports 95% Confidence Intervals for BOTH:
    1.  MICRO-AVERAGES (Overall pixel-level metrics) using a patient-level bootstrap.
    2.  MACRO-AVERAGES (Mean of per-image scores) using a score-level bootstrap.
    The AUC calculation is fully integrated.

    This version now correctly implements a patient-level (cluster) bootstrap.
    It includes a gatekeeper to only run the bootstrap if n_patients >= 20.
    """

    # --- 1. SETUP ---
    for model in models_list:
        model.eval()

    # --- 2. DETERMINE ENSEMBLE PARAMETERS ---
    if ENSEMBLE_METHOD == "WeightedAverage":
        print("\nUsing 'WeightedAverage' ensemble method.")
        model_weights = [m['ensemble_weight'] for m in constituent_models_info]
        print(f"Using global optimal threshold: {optimal_threshold:.4f}")

    elif ENSEMBLE_METHOD == "MajorityVote":
        print("\nUsing 'MajorityVote' ensemble method.")
        individual_thresholds = [float(m["optimal_threshold_optimized"]) for m in constituent_models_info]
        vote_thr = 0.5  # because we'll normalize weights to sum=1 in the loop
        print(f"Using weighted-majority threshold: {vote_thr:.2f} (>= 0.50 of total weight)")

    else:
        raise ValueError(f"Unknown ENSEMBLE_METHOD: '{ENSEMBLE_METHOD}'")

    # --- 2. DATA COLLECTION LOOP (PATIENT-AWARE) ---
    stats_by_patient = defaultdict(list) # Groups patch stats [{tp, fp, fn, tn}, ...] by patient_id
    auc_thresholds = np.linspace(0.0, 1.0, num_auc_steps)
    auc_pixel_counts = np.zeros((num_auc_steps, 4), dtype=np.int64) # Stores [tp, fp, fn, tn] for each threshold

    print(f"Running Ensemble test evaluation with optimal threshold: {optimal_threshold:.4f}")

    with torch.no_grad():
        pbar = tqdm(test_loader, desc="Ensemble Test", leave=False)
        for batch_data in pbar:
            if batch_data is None or batch_data[0] is None:
              continue
            images, masks, patient_ids = batch_data
            images=images.to(device, non_blocking=True)
            true_indices = torch.argmax(masks, dim=1).int()
            current_batch_size = images.size(0)
            if current_batch_size == 0: continue

            # --- Get individual model probabilities ---
            model_probs = []
            for model in models_list:
                probs_cancer = predict_with_tta(model, images)  # [B, H, W], float in [0,1]
                model_probs.append(probs_cancer)

            # ------------------------------------------------------------------
            # NEW (Option 2): Always compute a continuous probability score for AUC
            # using WeightedAverage, regardless of ENSEMBLE_METHOD
            # ------------------------------------------------------------------
            w = safe_tensor_weights(model_weights, device=model_probs[0].device, n_models=len(models_list))  # normalized

            # continuous score for ROC/AUC: [B,H,W]
            stack_probs = torch.stack(model_probs, dim=0)  # [M,B,H,W]
            ensemble_probs_for_auc = (stack_probs * w[:, None, None, None]).sum(dim=0).clamp(0, 1)

            # ------------------------------------------------------------------
            # Now compute the FINAL binary prediction mask depending on ENSEMBLE_METHOD
            # ------------------------------------------------------------------
            if ENSEMBLE_METHOD == "WeightedAverage":
                # use the recipe threshold on the weighted-average probs (same as AUC score)
                ensemble_probs_cancer = ensemble_probs_for_auc
                ensemble_preds_opt = (ensemble_probs_cancer >= optimal_threshold).int().cpu()

            elif ENSEMBLE_METHOD == "MajorityVote":
                # per-model thresholds (must exist in recipe)
                binary_masks = [(p >= t).float() for p, t in zip(model_probs, individual_thresholds)]  # each [B,H,W]
                stack_bin = torch.stack(binary_masks, dim=0)  # [M,B,H,W]

                # Weighted vote: sum(w_i * 1[p_i >= t_i])
                weighted_votes = (stack_bin * w[:, None, None, None]).sum(dim=0)  # [B,H,W], in [0,1] because w sums to 1

                # voting threshold:
                # - if you want strict "majority by weight": >= 0.5
                # - if you want a fixed k of M models: you'd use unweighted sum and compare to k
                vote_thr = 0.5
                ensemble_preds_opt = (weighted_votes >= vote_thr).int().cpu()

                # IMPORTANT: for AUC we do NOT use vote proportions anymore
                ensemble_probs_cancer = ensemble_probs_for_auc  # continuous, probability-like

            else:
                raise ValueError(f"Unknown ENSEMBLE_METHOD: '{ENSEMBLE_METHOD}'")

            true_indices_cpu = true_indices.cpu()

            # --- MODIFICATION: Group stats by patient ID ---
            for j in range(current_batch_size):
                pred_single, true_single = ensemble_preds_opt[j], true_indices_cpu[j]
                patient_id = patient_ids[j] # Get the patient ID for this image

                tp = ((pred_single == 1) & (true_single == 1)).sum().item()
                fp = ((pred_single == 1) & (true_single == 0)).sum().item()
                fn = ((pred_single == 0) & (true_single == 1)).sum().item()
                tn = ((pred_single == 0) & (true_single == 0)).sum().item()

                # Append the stats for this patch to the correct patient's list
                stats_by_patient[patient_id].append({'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn})

            # --- Accumulate counts for AUC curve (unchanged) ---
            ensemble_probs_cancer_cpu = ensemble_probs_for_auc.detach().cpu()
            for i, auc_thresh in enumerate(auc_thresholds):
                preds_binary_auc = (ensemble_probs_cancer_cpu >= auc_thresh).int()
                auc_pixel_counts[i, 0] += ((preds_binary_auc == 1) & (true_indices_cpu == 1)).sum().item() # TP
                auc_pixel_counts[i, 1] += ((preds_binary_auc == 1) & (true_indices_cpu == 0)).sum().item() # FP
                auc_pixel_counts[i, 2] += ((preds_binary_auc == 0) & (true_indices_cpu == 1)).sum().item() # FN
                auc_pixel_counts[i, 3] += ((preds_binary_auc == 0) & (true_indices_cpu == 0)).sum().item() # TN

            del images, masks, true_indices, ensemble_preds_opt, true_indices_cpu, ensemble_probs_cancer_cpu, model_probs

    # --- Post-Loop Calculations ---
    if not stats_by_patient:
        print("Error: No patients were processed successfully.")
        return None

    # --- ACTION POINT 2.1: Count unique patients ---
    unique_patient_ids = list(stats_by_patient.keys())
    n_patients = len(unique_patient_ids)

    # --- ACTION POINT 2.2: Implement the Conditional Gatekeeper ---
    run_bootstrap = n_patients >= 20

    if not run_bootstrap:
        print("\n" + "="*80)
        print(f"n_patients: {n_patients}")
        print("WARNING: You cannot run bootstrap strategy with accuracy for a cluster with less than 20 patients minimum.")
        print("Script will only calculate the metrics with no confidence intervals.")
        print("="*80 + "\n")


    # Define metric keys and bootstrap samples (will only be used if run_bootstrap is True)
    n_bootstrap_samples = 10000
    metric_keys = ['dice', 'iou', 'accuracy', 'tpr', 'tnr', 'precision', 'fpr', 'fnr']

    # --- Initialize results dictionaries ---
    # These will be populated differently depending on the bootstrap condition.
    final_micro_metrics = {'point_estimate': {}, 'ci': {key: [np.nan, np.nan] for key in metric_keys}}
    final_macro_metrics = {'point_estimate': {}, 'ci': {key: [np.nan, np.nan] for key in metric_keys}}

    # --- 4. POINT ESTIMATE CALCULATIONS (RUN ALWAYS) ---
    print("\nCalculating Point Estimate Metrics from the full test set...")

    # Micro-Averages (from all patches)
    all_patches_stats = [stat for pat_stats in stats_by_patient.values() for stat in pat_stats]
    tp_micro, fp_micro, fn_micro, tn_micro = np.sum([list(s.values()) for s in all_patches_stats], axis=0)
    final_micro_metrics['point_estimate'] = calculate_metrics(tp_micro, fp_micro, fn_micro, tn_micro)

    # Macro-Averages (mean of per-patient scores)
    per_patient_scores = {key: np.zeros(n_patients) for key in metric_keys}
    for i, patient_id in enumerate(unique_patient_ids):
        tp_pat, fp_pat, fn_pat, tn_pat = np.sum([list(s.values()) for s in stats_by_patient[patient_id]], axis=0)
        patient_metrics = calculate_metrics(tp_pat, fp_pat, fn_pat, tn_pat)
        for key in metric_keys:
            per_patient_scores[key][i] = patient_metrics[key]

    for metric in metric_keys:
        final_macro_metrics['point_estimate'][metric] = np.nanmean(per_patient_scores[metric])

    # --- 5. CONDITIONAL BOOTSTRAP FOR CONFIDENCE INTERVALS ---
    if run_bootstrap:
        print(f"\nCalculating 95% CIs with Patient-Level Bootstrap ({n_bootstrap_samples} resamples)...")
        np.random.seed(SEED) # For reproducibility

        # Bootstrap for MICRO-AVERAGED Metrics
        bootstrap_micro_metrics = {key: np.zeros(n_bootstrap_samples) for key in metric_keys}
        for i in tqdm(range(n_bootstrap_samples), desc="Bootstrap (Micro)", leave=False):
            resampled_patient_ids = np.random.choice(unique_patient_ids, size=n_patients, replace=True)
            bootstrap_stats_sample = [stats_by_patient[pid] for pid in resampled_patient_ids]
            all_bootstrap_patches = [stat for pat_stats in bootstrap_stats_sample for stat in pat_stats]
            tp, fp, fn, tn = np.sum([list(s.values()) for s in all_bootstrap_patches], axis=0)
            metrics = calculate_metrics(tp, fp, fn, tn)
            for key in metric_keys:
                bootstrap_micro_metrics[key][i] = metrics[key]
        for metric in metric_keys:
            final_micro_metrics['ci'][metric] = np.percentile(bootstrap_micro_metrics[metric], [2.5, 97.5])

        # ACTION 2.3: Bootstrap for MACRO-AVERAGED Metrics (Use np.nanmean)
        bootstrap_macro_means = {key: np.zeros(n_bootstrap_samples) for key in metric_keys}
        for i in tqdm(range(n_bootstrap_samples), desc="Bootstrap (Macro)", leave=False):
            # Resample patient-level scores with replacement
            indices = np.random.choice(range(n_patients), size=n_patients, replace=True)
            for metric in metric_keys:
                 # Use nanmean to correctly average the resampled scores, ignoring NaNs
                 bootstrap_macro_means[metric][i] = np.nanmean(per_patient_scores[metric][indices])

        for metric in metric_keys:
            # The percentile calculation is robust to NaNs if they were filtered by nanmean
            final_macro_metrics['ci'][metric] = np.percentile(bootstrap_macro_means[metric], [2.5, 97.5])

    # --- 6. AUC CALCULATION ---
    auc_score = 0.0
    print("\nCalculating Ensemble AUC from accumulated counts...")
    try:
        total_pos = auc_pixel_counts[0, 0] + auc_pixel_counts[0, 2] # Initial TP + FN
        total_neg = auc_pixel_counts[0, 1] + auc_pixel_counts[0, 3] # Initial FP + TN
        if total_pos > 0 and total_neg > 0:
            tpr_values = auc_pixel_counts[:, 0] / total_pos
            fpr_values = auc_pixel_counts[:, 1] / total_neg
            order = np.argsort(fpr_values)
            auc_score = sklearn_auc(fpr_values[order], tpr_values[order])
    except Exception as e:
        print(f"AUC Calculation Error: {e}")


    # --- 7. FINAL REPORTING (FLEXIBLE) ---
    print("\n" + "="*25 + " Final Performance Summary " + "="*25)
    print("\n--- Tumor Segmentation Metrics (Micro, Pixel-Level) ---")
    print("  Note: Dice/IoU are computed for the tumor class (positive pixels).")
    for metric in sorted(metric_keys):
        print_metric_line(metric, final_micro_metrics['point_estimate'][metric], final_micro_metrics['ci'][metric])
    print(f"  {'AUC':<12}: {auc_score:.4f}  (CI not calculated via bootstrap)")

    print("\n--- Tumor Segmentation Metrics (Macro, Patient-Level) ---")
    print("  Note: Per-patient metrics are aggregated over tumor patients only (negatives give NaN).")
    for metric in sorted(metric_keys):
        print_metric_line(metric, final_macro_metrics['point_estimate'][metric], final_macro_metrics['ci'][metric])


    return {
        "micro_averaged_metrics": final_micro_metrics,   # contains point_estimate + ci
        "macro_averaged_metrics": final_macro_metrics,   # contains point_estimate + ci
        "auc": float(auc_score),

        # --- NEW: what you need for the paper ---
        "bootstrap": {
            "ran": bool(run_bootstrap),
            "n_patients": int(n_patients),
            "n_bootstrap_samples": int(n_bootstrap_samples) if run_bootstrap else 0,
            "seed": int(SEED),
        },
        "confusion_matrix": {
            "tp": int(tp_micro),
            "fp": int(fp_micro),
            "fn": int(fn_micro),
            "tn": int(tn_micro),
        },
        "normalization": {
            "mean": [float(x) for x in train_mean],   # pass train_mean into analyze_ensemble_metrics OR set global
            "std":  [float(x) for x in train_std],
            "computed_from": "TRAIN set (post-stain-normalization)",
            "stats_sample_size": "ALL",  # or whatever you used (store your actual variable)
        },
        "ensemble": {
            "method": str(ENSEMBLE_METHOD),
            "threshold": float(optimal_threshold),
            "weights": [float(x) for x in model_weights] if model_weights is not None else None,
            "vote_thr": 0.5 if ENSEMBLE_METHOD == "MajorityVote" else None,
            "per_model_thresholds": (
                [float(m["optimal_threshold_optimized"]) for m in constituent_models_info]
                if ENSEMBLE_METHOD == "MajorityVote" else None
            ),
        },
    }

In [ ]:
def visualize_ensemble_predictions(
    models_list,
    model_weights,
    dataloader,
    device,
    threshold,
    num_samples,
    train_mean,
    train_std,
    constituent_models_info=None,
    ensemble_method=None,
    vote_thr=0.5,
    show_prob_heatmap=True,
):
    """
    Visualize ensemble predictions consistently with the evaluation logic.

    - WeightedAverage:
        pred = (sum_i w_i * p_i) >= threshold

    - MajorityVote (weighted-majority):
        pred = (sum_i w_i * 1[p_i >= t_i]) >= vote_thr
        where vote_thr=0.5 assumes weights are normalized to sum=1.

    Notes:
    - constituent_models_info is REQUIRED for MajorityVote to get per-model thresholds.
      It must contain "optimal_threshold_optimized" for each model.
    - model_weights will be normalized inside this function.
    """

    if ensemble_method is None:
        ensemble_method = ENSEMBLE_METHOD  # falls back to your global

    num_models = len(models_list)
    if num_models == 0:
        print("No models provided.")
        return

    # Normalize weights safely
    if model_weights is None:
        w = torch.ones(num_models, device=device, dtype=torch.float32) / float(num_models)
        w_cpu = (w.detach().cpu().numpy()).tolist()
    else:
        w = torch.tensor(model_weights, device=device, dtype=torch.float32)
        w = w / (w.sum() + 1e-12)
        w_cpu = (w.detach().cpu().numpy()).tolist()

    # MajorityVote: fetch per-model thresholds
    individual_thresholds = None
    if ensemble_method == "MajorityVote":
        if not constituent_models_info:
            raise ValueError("MajorityVote requires constituent_models_info to get per-model thresholds.")
        individual_thresholds = [float(m["optimal_threshold_optimized"]) for m in constituent_models_info]
        if len(individual_thresholds) != num_models:
            raise ValueError("Length of individual_thresholds must match number of models.")
        print(f"[Viz] MajorityVote enabled. Weighted vote threshold={vote_thr:.2f}, per-model thresholds loaded.")

    for model in models_list:
        model.eval()

    print(f"Visualizing {num_samples} ENSEMBLE samples | Method={ensemble_method} | thr={threshold:.4f}")

    # Grab a single batch
    with torch.no_grad():
        try:
            batch_data = next(iter(dataloader))
        except StopIteration:
            print("DataLoader empty.")
            return

        if batch_data is None:
            print("Cannot load batch (collate_fn returned None).")
            return

        images, masks, patient_ids = batch_data
        if images is None or images.shape[0] == 0:
            print("Empty image batch.")
            return

        actual_batch_size = images.shape[0]
        num_samples = min(num_samples, actual_batch_size)
        sample_indices = random.sample(range(actual_batch_size), num_samples)

        images_vis = images[sample_indices].to(device, non_blocking=True)
        masks_vis = masks[sample_indices]  # keep on CPU
        true_classes = torch.argmax(masks_vis, dim=1).numpy().astype(np.uint8)  # [N,H,W]

        # --- Compute per-model probabilities on these samples ---
        model_probs = []
        for model in models_list:
            probs_cancer = predict_with_tta(model, images_vis)  # [N,H,W], float [0,1]
            model_probs.append(probs_cancer)

        # Stack to [M,N,H,W]
        stack_probs = torch.stack(model_probs, dim=0)

        # Always compute a continuous ensemble probability score (useful for heatmaps)
        ensemble_probs_for_auc = (stack_probs * w[:, None, None, None]).sum(dim=0).clamp(0, 1)  # [N,H,W]

        # --- Produce final binary predictions consistent with evaluation ---
        if ensemble_method == "WeightedAverage":
            ensemble_probs_cancer = ensemble_probs_for_auc  # same definition
            ensemble_preds = (ensemble_probs_cancer >= float(threshold)).to(torch.uint8).cpu().numpy()

        elif ensemble_method == "MajorityVote":
            # Per-model binarization with per-model thresholds
            bin_stack = []
            for p, t in zip(model_probs, individual_thresholds):
                bin_stack.append((p >= t).float())
            bin_stack = torch.stack(bin_stack, dim=0)  # [M,N,H,W]

            # Weighted vote: sum(w_i * bin_i) in [0,1] because w sums to 1
            weighted_votes = (bin_stack * w[:, None, None, None]).sum(dim=0)  # [N,H,W]

            ensemble_preds = (weighted_votes >= float(vote_thr)).to(torch.uint8).cpu().numpy()

            # what to visualize as "probability heatmap"
            ensemble_probs_cancer = ensemble_probs_for_auc if show_prob_heatmap else weighted_votes

        else:
            raise ValueError(f"Unknown ensemble_method: {ensemble_method}")

        # --- Prepare images for plotting (de-normalize) ---
        images_np = images_vis.detach().cpu().numpy()  # [N,C,H,W]
        mean = np.array(train_mean, dtype=np.float32)
        std = np.array(train_std, dtype=np.float32)

        for j in range(num_samples):
            img = images_np[j].transpose(1, 2, 0)  # HWC
            img = std * img + mean
            img = np.clip(img, 0, 1)

            pred_mask = ensemble_preds[j]
            true_mask = true_classes[j]

            fig, axes = plt.subplots(1, 4, figsize=(20, 5))

            axes[0].imshow(img)
            axes[0].set_title("Image")
            axes[0].axis("off")

            # Pred overlay
            axes[1].imshow(img)
            axes[1].imshow(np.ma.masked_where(pred_mask == 0, pred_mask), cmap="jet", alpha=0.5)
            axes[1].set_title(f"Ensemble Pred (method={ensemble_method})")
            axes[1].axis("off")

            # True overlay
            axes[2].imshow(img)
            axes[2].imshow(np.ma.masked_where(true_mask == 0, true_mask), cmap="jet", alpha=0.5)
            axes[2].set_title("True Mask")
            axes[2].axis("off")

            # Probability / score heatmap
            prob_map = ensemble_probs_cancer[j].detach().cpu().numpy()
            axes[3].imshow(prob_map, vmin=0, vmax=1)
            axes[3].set_title("Ensemble Score (0..1)")
            axes[3].axis("off")

            plt.tight_layout()
            plt.show()

In [ ]:
def print_scientific_analysis_report(metrics_results):
    """
    Prints a detailed, objective, and scientific guide for interpreting the model's
    performance, now including explanations for patient-level bootstrapping and CIs.
    """
    if not metrics_results:
        print("\nMetrics object is empty. Cannot generate report.")
        return

    # --- Unpack all the necessary data ---
    micro = metrics_results.get('micro_averaged_metrics', {})
    macro = metrics_results.get('macro_averaged_metrics', {})
    auc = metrics_results.get('auc')

    if not micro or not macro or auc is None:
        print("\nMetrics object is missing required data. Cannot generate full report.")
        return

    micro_pe = micro.get('point_estimate', {})
    micro_ci = micro.get('ci', {})
    macro_pe = macro.get('point_estimate', {})
    macro_ci = macro.get('ci', {})

    # --- Helper to format the CI string dynamically ---
    def get_ci_string(ci_data):
        if ci_data is not None and not np.isnan(ci_data[0]):
            return f"(95% CI: [{ci_data[0]:.4f}, {ci_data[1]:.4f}])"
        else:
            return "(CI not calculated; n_patients < 20)"

    print("\n" + "="*30 + " Scientific Performance Analysis " + "="*30)
    print("\nThis report provides a scientific context for the model's performance on the hold-out test set.")
    print("It emphasizes patient-level generalization and statistical uncertainty.")

    print("\n\n" + "-"*25 + " Analysis of Key Metrics " + "-"*25)

    # --- 1. Segmentation Quality (Dice & IoU) ---
    print("\n[--- Segmentation Quality: Dice and IoU ---]")
    print("  - Definition: Measures the spatial overlap between predicted and true masks (Ideal = 1.0).")
    print("  - Micro-Average: Aggregates all pixels from all patients. This reflects overall pixel-level accuracy")
    print("    but can be dominated by patients who contributed a large number of patches.")
    print(f"    - Micro-Average Dice: {micro_pe.get('dice', 0):.4f} {get_ci_string(micro_ci.get('dice'))}")

    print("\n  - Macro-Average: Calculates the metric for each patient first, then averages these scores. This is the")
    print("    primary metric for clinical generalization, as it treats each patient equally.")
    print(f"    - Macro-Average Dice (Per-Patient Mean): {macro_pe.get('dice', 0):.4f} {get_ci_string(macro_ci.get('dice'))}")

    print("\n  - Interpretation of the 95% Confidence Interval (CI): The CI provides a plausible range for the true")
    print("    performance metric. A narrow CI suggests that the model's performance is stable and consistent")
    print("    across different patients in the test set.")

    # --- 2. Clinical Reliability: Sensitivity (TPR) and Miss Rate (FNR) ---
    print("\n[--- Clinical Reliability: Sensitivity / Miss Rate ---]")
    print("  - Definition (FNR): The False Negative Rate, or 'Miss Rate' (Ideal = 0.0). It is the proportion of")
    print("    cancerous regions/patients that the model failed to detect.")
    print("  - Interpretation: This metric is critical for clinical safety. A low FNR is essential for a screening tool.")
    print(f"  - The model's Macro-Average FNR is {macro_pe.get('fnr', 0):.4f} {get_ci_string(macro_ci.get('fnr'))}. This suggests that, on average,")
    print(f"    the model is expected to miss approximately {macro_pe.get('fnr', 0):.2%} of cancerous patients/regions.")

    # --- 3. Clinical Reliability: Specificity (TNR) and False Alarm Rate (FPR) ---
    print("\n[--- Clinical Reliability: Specificity / False Alarm Rate ---]")
    print("  - Definition (FPR): The False Positive Rate, or 'False Alarm Rate' (Ideal = 0.0). It is the proportion")
    print("    of healthy regions/patients that were incorrectly flagged as cancerous.")
    print("  - Interpretation: This metric is important for clinical efficiency, reducing unnecessary reviews.")
    print(f"  - The model's Macro-Average FPR is {macro_pe.get('fpr', 0):.4f} {get_ci_string(macro_ci.get('fpr'))}. This suggests that, on average,")
    print(f"    an estimated {macro_pe.get('fpr', 0):.2%} of non-cancerous patients/regions would trigger a false alarm.")

    # --- 4. Overall Discriminative Power (AUC) ---
    print("\n[--- Overall Discriminative Power: AUC ---]")
    print("  - Definition: The Area Under the ROC Curve measures the model's ability to distinguish between")
    print("    positive and negative pixels across all possible thresholds (Ideal = 1.0).")
    print(f"  - The model's pixel-level AUC is {auc:.4f}. As a benchmark, values above 0.95 typically reflect")
    print("    excellent discriminative power between the classes.")

    print("\n\n" + "="*25 + " How to Form a Conclusion " + "="*25)
    print("A robust and generalizable model demonstrates a combination of strengths:")
    print("  1. High Technical Skill: Indicated by a high Micro-Average Dice and a high AUC.")
    print("  2. High Generalization to New Patients: Indicated by a strong Macro-Average (per-patient) Dice score.")
    print("  3. High Safety & Sensitivity: Indicated by a low Macro-Average FNR.")
    print("  4. High Efficiency & Specificity: Indicated by a low Macro-Average FPR.")
    print("  5. High Confidence: Indicated by narrow 95% Confidence Intervals on the key macro-average metrics.")
    print("\nEvaluate these metrics based on the intended clinical application. For a screening tool, a low")
    print("FNR and its upper CI bound are paramount. For a confirmatory tool, a low FPR may be more critical.")
    print("="*80)

In [ ]:
# You can add this function definition after the `print_scientific_analysis_report` function

def export_results_to_csv(ensemble_recipe, metrics_results, output_dir):
    """
    Exports the complete configuration and final results of the ensemble evaluation
    to a comprehensive and machine-readable CSV file.
    """
    if not metrics_results or not ensemble_recipe:
        print("Cannot export results to CSV: Missing metrics or ensemble recipe.")
        return

    timestamp = get_formatted_datetime_string()
    filename = f"FINAL_EVALUATION_REPORT_{timestamp}.csv"
    filepath = os.path.join(output_dir, filename)

    print(f"\n--- Exporting final report to CSV: {filepath} ---")

    # --- 1. Prepare Data for Export ---

    # Section 1: Experiment Configuration
    config_data = {
        'Parameter': [
            'Experiment Datetime',
            'Ensemble Recipe Path',
            'Dataset ZIP Directory',
            'Evaluation Set',
            'Random Seed',
            'Batch Size',
            'Optimal Ensemble Threshold'
        ],
        'Value': [
            timestamp,
            ENSEMBLE_META_PATH, # Global variable
            DATASET_ZIP_DIR,    # Global variable
            'TEST',
            SEED,               # Global variable
            BATCH_SIZE,         # Global variable
            ensemble_recipe.get('opt_pool_threshold')
        ]
    }
    config_df = pd.DataFrame(config_data)

    # Section 2: Ensemble Composition
    comp_models = ensemble_recipe.get('constituent_models', [])
    composition_data = {
        'Model Index': [f"Model {i+1}" for i in range(len(comp_models))],
        'Architecture': [m.get('architecture') for m in comp_models],
        'Encoder': [m.get('encoder') for m in comp_models],
        'Ensemble Weight': [m.get('ensemble_weight') for m in comp_models],
        'Source Checkpoint': [os.path.basename(m.get('checkpoint_path', '')) for m in comp_models]
    }
    composition_df = pd.DataFrame(composition_data)

    # Section 3: Performance Metrics
    micro = metrics_results.get('micro_averaged_metrics', {})
    macro = metrics_results.get('macro_averaged_metrics', {})
    auc = metrics_results.get('auc')

    metric_keys = sorted(micro.get('point_estimate', {}).keys())

    metrics_data = {
        'Metric Type': [],
        'Metric': [],
        'Point Estimate': [],
        'CI Lower (2.5%)': [],
        'CI Upper (97.5%)': []
    }

    # Add Micro-Averages
    for key in metric_keys:
        metrics_data['Metric Type'].append('Micro-Average (Pixel Level)')
        metrics_data['Metric'].append(key.title())
        metrics_data['Point Estimate'].append(micro.get('point_estimate', {}).get(key))
        ci = micro.get('ci', {}).get(key, [np.nan, np.nan])
        metrics_data['CI Lower (2.5%)'].append(ci[0])
        metrics_data['CI Upper (97.5%)'].append(ci[1])

    # Add AUC
    metrics_data['Metric Type'].append('Micro-Average (Pixel Level)')
    metrics_data['Metric'].append('AUC')
    metrics_data['Point Estimate'].append(auc)
    metrics_data['CI Lower (2.5%)'].append(np.nan)
    metrics_data['CI Upper (97.5%)'].append(np.nan)

    # Add Macro-Averages
    for key in metric_keys:
        metrics_data['Metric Type'].append('Macro-Average (Patient Level)')
        metrics_data['Metric'].append(key.title())
        metrics_data['Point Estimate'].append(macro.get('point_estimate', {}).get(key))
        ci = macro.get('ci', {}).get(key, [np.nan, np.nan])
        metrics_data['CI Lower (2.5%)'].append(ci[0])
        metrics_data['CI Upper (97.5%)'].append(ci[1])

    metrics_df = pd.DataFrame(metrics_data)

    # --- 2. Write to CSV File ---
    try:
        with open(filepath, 'w', newline='') as f:
            f.write("--- Experiment Configuration ---\n")
            config_df.to_csv(f, index=False)

            f.write("\n--- Ensemble Composition ---\n")
            composition_df.to_csv(f, index=False)

            f.write("\n--- Final Performance Metrics ---\n")
            metrics_df.to_csv(f, index=False)

        print(f"Successfully exported detailed results to {filepath}")
    except Exception as e:
        print(f"Failed to export results to CSV: {e}")

In [ ]:
def parse_ensemble_recipe(recipe: dict):
    if "opt_pool_threshold" in recipe:
        thr = float(recipe["opt_pool_threshold"])
    else:
        raise KeyError("Could not find ensemble threshold in recipe. Expected "
                       "'opt_pool_threshold'.")

    # --- constituent models ---
    models = recipe.get("constituent_models", None)
    if not models:
        raise KeyError("Recipe missing 'constituent_models'.")

    # --- weights ---
    weights = []
    for m in models:
        if "ensemble_weight" not in m:
            raise KeyError("A constituent model is missing 'ensemble_weight'.")
        weights.append(float(m["ensemble_weight"]))

    # normalize weights (safe)
    s = sum(weights)
    if s <= 0:
        weights = [1.0 / len(weights)] * len(weights)
    else:
        weights = [w / s for w in weights]

    return thr, models, weights

In [ ]:
def load_checkpoint_strict_without_aux(model, chkpt_path, device):
    chkpt = torch.load(chkpt_path, map_location="cpu")
    state_dict = chkpt.get("model_state_dict", chkpt)

    # Drop aux head keys if present
    state_dict = {k: v for k, v in state_dict.items() if not k.startswith("classification_head.")}

    # Handle torch.compile prefix mismatch
    model_keys = list(model.state_dict().keys())
    sd_keys = list(state_dict.keys())
    if model_keys and sd_keys:
        model_has = model_keys[0].startswith("_orig_mod.")
        sd_has = sd_keys[0].startswith("_orig_mod.")
        if model_has and not sd_has:
            state_dict = {f"_orig_mod.{k}": v for k, v in state_dict.items()}
        elif (not model_has) and sd_has:
            state_dict = {k.replace("_orig_mod.", "", 1): v for k, v in state_dict.items()}

    model.load_state_dict(state_dict, strict=True)
    model.to(device)

    if chkpt.get("is_compiled", False):
        print("[INFO] Re-compiling model (checkpoint was compiled)")
        try:
            model = torch.compile(model)
        except Exception as e:
            print(f"[WARN] torch.compile failed: {e}")

    return model

In [ ]:
def save_confusion_matrix_png(tp, fp, fn, tn, out_path_png):
    """
    Saves row-normalized confusion matrix as a PNG suitable for papers.
    """
    cm = np.array([[tn, fp], [fn, tp]], dtype=np.float64)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt='.2%',
        cmap='Blues',
        xticklabels=['Pred NoCancer', 'Pred Cancer'],
        yticklabels=['True NoCancer', 'True Cancer']
    )
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Ensemble Confusion Matrix (Row-Normalized %)')
    plt.tight_layout()
    plt.savefig(out_path_png, dpi=300, bbox_inches="tight")
    plt.close()
    return out_path_png

In [ ]:
def _tex_escape(s: str) -> str:
    # minimal escape for paths / underscores
    return (s.replace('\\', r'\textbackslash ')
             .replace('_', r'\_')
             .replace('%', r'\%')
             .replace('&', r'\&')
             .replace('#', r'\#')
             .replace('{', r'\{')
             .replace('}', r'\}'))

def fmt_ci(ci_pair, ran_bootstrap: bool):
    """
    Return CI string: [low, high] if bootstrap ran and CI is valid, else NA.
    """
    if not ran_bootstrap:
        return "NA"
    if ci_pair is None:
        return "NA"
    lo, hi = ci_pair
    if (lo is None) or (hi is None) or np.isnan(lo) or np.isnan(hi):
        return "NA"
    return f"[{lo:.4f}, {hi:.4f}]"

def fmt_float(x):
    if x is None:
        return "NA"
    try:
        if np.isnan(x):
            return "NA"
    except Exception:
        pass
    return f"{float(x):.4f}"

def write_ensemble_report_latex(
    ensemble_recipe: dict,
    ensemble_metrics: dict,
    train_mean, train_std,
    stats_sample_size: int,
    cm_png_path: str,
    output_dir: str,
    report_name_prefix: str = "FINAL_ENSEMBLE_REPORT"
):
    """
    Writes report.tex and compiles to report.pdf in output_dir.
    Includes:
      - micro & macro metrics with point estimate + CI/NA
      - normalization mean/std
      - confusion matrix figure
      - ensemble composition table
    """
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
    tex_path = os.path.join(output_dir, f"{report_name_prefix}_{ts}.tex")
    pdf_path = os.path.join(output_dir, f"{report_name_prefix}_{ts}.pdf")

    # --- Pull bootstrap info safely ---
    bootstrap = ensemble_metrics.get("bootstrap", {})
    ran_bootstrap = bool(bootstrap.get("ran", False))
    n_patients = bootstrap.get("n_patients", None)
    n_boot = bootstrap.get("n_bootstrap_samples", None)
    boot_seed = bootstrap.get("seed", None)

    micro = ensemble_metrics.get("micro_averaged_metrics", {})
    macro = ensemble_metrics.get("macro_averaged_metrics", {})
    auc = ensemble_metrics.get("auc", None)

    micro_pe = micro.get("point_estimate", {})
    micro_ci = micro.get("ci", {})
    macro_pe = macro.get("point_estimate", {})
    macro_ci = macro.get("ci", {})

    # Metrics order (paper-friendly)
    metric_order = ["dice", "iou", "tpr", "tnr", "precision", "accuracy", "fpr", "fnr"]

    # Ensemble composition
    comp_models = ensemble_recipe.get("constituent_models", [])
    # Try threshold key variants
    thr = ensemble_recipe.get("opt_pool_threshold", ensemble_recipe.get("optimal_ensemble_threshold", None))

    # --- LaTeX content ---
    tex = rf"""
\documentclass[11pt]{{article}}
\usepackage[a4paper,margin=1in]{{geometry}}
\usepackage{{booktabs}}
\usepackage{{graphicx}}
\usepackage{{float}}
\usepackage{{caption}}
\usepackage{{amsmath}}
\usepackage{{array}}
\usepackage{{hyperref}}

\title{{Ensemble Final Evaluation Report}}
\author{{Automated Pipeline}}
\date{{{ts}}}

\begin{{document}}
\maketitle

\section*{{Experiment Summary}}
\begin{{itemize}}
  \item Objective (from recipe): {_tex_escape(str(ensemble_recipe.get("optuna_objective_name", "NA")))}
  \item Threshold (recipe): {fmt_float(thr)}
  \item Bootstrap ran: {"Yes" if ran_bootstrap else "No"} \\
        (n\_patients={_tex_escape(str(n_patients))}, n\_bootstrap={_tex_escape(str(n_boot))}, seed={_tex_escape(str(boot_seed))})
  \item AUC (pixel-level ROC): {fmt_float(auc)}
\end{{itemize}}

\section*{{Normalization Statistics}}
Computed on TRAIN set (post-stain-normalization), then applied to TEST.
\begin{{itemize}}
  \item Mean (RGB): [{train_mean[0]:.6f}, {train_mean[1]:.6f}, {train_mean[2]:.6f}]
  \item Std (RGB): [{train_std[0]:.6f}, {train_std[1]:.6f}, {train_std[2]:.6f}]
  \item Stats sample size: {_tex_escape(str(stats_sample_size))}
\end{{itemize}}

\section*{{Ensemble Composition}}
\begin{{table}}[H]
\centering
\caption{{Constituent models and weights (from recipe).}}
\begin{{tabular}}{{r l l r}}
\toprule
\# & Architecture & Encoder & Weight \\
\midrule
"""
    # rows
    for i, m in enumerate(comp_models, start=1):
        arch = _tex_escape(str(m.get("architecture", "NA")))
        enc  = _tex_escape(str(m.get("encoder", "NA")))
        w    = m.get("ensemble_weight", None)
        tex += rf"{i} & {arch} & {enc} & {fmt_float(w)} \\ " + "\n"

    tex += r"""
\bottomrule
\end{tabular}
\end{table}
"""

    # Micro table
    tex += r"""
\section*{Final Metrics (Micro / Pixel-Level)}
\begin{table}[H]
\centering
\caption{Micro-averaged metrics with 95\% CI (NA if bootstrap did not run).}
\begin{tabular}{l r l}
\toprule
Metric & Point Estimate & 95\% CI \\
\midrule
"""
    for k in metric_order:
        pe = micro_pe.get(k, None)
        ci = micro_ci.get(k, None)
        tex += rf"{_tex_escape(k.upper())} & {fmt_float(pe)} & {fmt_ci(ci, ran_bootstrap)} \\ " + "\n"
    tex += r"""
\bottomrule
\end{tabular}
\end{table}
"""

    # Macro table
    tex += r"""
\section*{Final Metrics (Macro / Patient-Level)}
\begin{table}[H]
\centering
\caption{Macro-averaged (per-patient) metrics with 95\% CI (NA if bootstrap did not run).}
\begin{tabular}{l r l}
\toprule
Metric & Point Estimate & 95\% CI \\
\midrule
"""
    for k in metric_order:
        pe = macro_pe.get(k, None)
        ci = macro_ci.get(k, None)
        tex += rf"{_tex_escape(k.upper())} & {fmt_float(pe)} & {fmt_ci(ci, ran_bootstrap)} \\ " + "\n"
    tex += r"""
\bottomrule
\end{tabular}
\end{table}
"""

    # Confusion matrix figure
    cm_png_rel = os.path.basename(cm_png_path)
    tex += rf"""
\section*{{Confusion Matrix}}
\begin{{figure}}[H]
\centering
\includegraphics[width=0.75\linewidth]{{{_tex_escape(cm_png_rel)}}}
\caption{{Row-normalized confusion matrix (percent).}}
\end{{figure}}
"""

    tex += r"""
\end{document}
"""

    # --- Write .tex and copy cm png to same folder (so LaTeX finds it) ---
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(tex)

    # Ensure figure is in same dir as tex
    cm_target = os.path.join(output_dir, os.path.basename(cm_png_path))
    if os.path.abspath(cm_png_path) != os.path.abspath(cm_target):
        import shutil
        shutil.copy2(cm_png_path, cm_target)

    # --- Compile to PDF (Colab has pdflatex usually; if not, you'll see the error) ---
    try:
        subprocess.run(
            ["pdflatex", "-interaction=nonstopmode", os.path.basename(tex_path)],
            cwd=output_dir,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        # pdflatex outputs PDF with same basename as tex
        built_pdf = os.path.join(output_dir, os.path.splitext(os.path.basename(tex_path))[0] + ".pdf")
        if os.path.exists(built_pdf):
            os.rename(built_pdf, pdf_path)
        print(f"[OK] PDF generated: {pdf_path}")
    except Exception as e:
        print("[WARN] pdflatex failed. The .tex was still created so you can compile manually.")
        print(f"       TeX file: {tex_path}")
        print(f"       Error: {e}")

    return tex_path, pdf_path

In [ ]:
# ==============================================================================
# --- 13. Main Training Loop ---
# ==============================================================================
print(f"\n{'='*25} Starting Main Training Process {'='*25}")

fold_zip_filename = f'MASTER_SET_1.zip'
fold_zip_path = os.path.join(DATASET_ZIP_DIR, fold_zip_filename)

# --- Extract Dataset ---
print(f"Extracting Fold...")
if not os.path.exists(fold_zip_path):
  print(f"Zip not found: {fold_zip_path}. Skip.")
  sys.exit(0)
try:
    if os.path.exists(base_data_dir):
      shutil.rmtree(base_data_dir)
    os.makedirs(base_data_dir, exist_ok=True);

    with zipfile.ZipFile(fold_zip_path,'r') as z:
      z.extractall(base_data_dir)

    print("Extracted. Verifying...");

    if not os.path.isdir(train_cancer_image_dir) or not os.listdir(train_cancer_image_dir):
      raise RuntimeError("Verify failed")

    print("Verified.")

except Exception as e:
  print(f"Extract Err: {e}. Skip.");
  clear_gpu();

In [ ]:
# --- DataLoaders (with optional stratified subsampling) ---
print("\nCreating DataLoaders...")

mean_to_be_used = MEAN if (MEAN is not None and len(MEAN) > 0) else None
std_to_be_used = STD if (STD is not None and len(STD) > 0) else None

try:
    # Always instantiate the full datasets first
    # 1) TRAIN: compute dataset-specific stats on full TRAIN
    full_train_ds = ProstateCancerDataset(
        train_cancer_image_dir,
        train_cancer_mask_dir,
        train_not_cancer_image_dir,
        train_not_cancer_mask_dir,
        compute_stats=False,         # <--- compute on full TRAIN
        stats_sample_size=None,
        mean = mean_to_be_used,
        std = std_to_be_used
    )

    train_mean = full_train_ds.mean if len(MEAN)==0 else MEAN
    train_std  = full_train_ds.std if len(STD)==0 else STD


    # 2) VAL: reuse TRAIN stats (no compute_stats here!)
    full_test_ds = ProstateCancerDataset(
        test_cancer_image_dir,
        test_cancer_mask_dir,
        test_not_cancer_image_dir,
        test_not_cancer_mask_dir,
        mean=train_mean,
        std=train_std,
    )

    # --- NEW: Print "Before" counts ---
    test_counts_before = full_test_ds.get_class_counts()

    print("\n--- Full Dataset Class Counts (Before Subsampling) ---")
    print(f" TEST: CANCER={test_counts_before['CANCER']}, NOT_CANCER={test_counts_before['NOT_CANCER']}")
    print('-'*50)

    def collate_fn(batch):
      batch = list(filter(lambda x: x is not None and x[0] is not None, batch))
      return torch.utils.data.dataloader.default_collate(batch) if batch else None

    train_ds = full_train_ds
    test_ds = full_test_ds


    test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=True, collate_fn=collate_fn)

    print(f"Test dataset size: {len(test_ds)}")
    print("DataLoaders created successfully.")

except Exception as e:
  print(f"DataLoader Err: {e}. Skip.")
  clear_gpu()

In [ ]:
# ### REFACTORED: Step 1 - Load the pre-computed ensemble recipe (NEW, IMPROVED VERSION) ###
print(f"\nLoading ensemble recipe from: {ENSEMBLE_META_PATH}")
if not os.path.exists(ENSEMBLE_META_PATH):
    raise FileNotFoundError(f"Ensemble metadata file not found! Path: {ENSEMBLE_META_PATH}")

with open(ENSEMBLE_META_PATH, 'r') as f:
    ensemble_recipe = json.load(f)

In [ ]:
optimal_threshold, constituent_models_info, optimal_weights = parse_ensemble_recipe(ensemble_recipe)
n_models = len(constituent_models_info)

print(f"Using Optimal Threshold: {optimal_threshold:.6f}")
print("Using Optimal Weights:")
for i, m in enumerate(constituent_models_info):
    print(f"  - {optimal_weights[i]:.6f} -> {m['architecture']} ({m['encoder']})")

In [ ]:
# ### REFACTORED: Step 2 - Load the EXACT models from the recipe ###
ensemble_models = []
print(f"\nLoading the {n_models} constituent models...")
for i, model_meta in enumerate(constituent_models_info):
    arch = model_meta.get('architecture')
    enc = model_meta.get('encoder')
    chkpt_path = model_meta.get('checkpoint_path')
    chkpt_path = change_paths(chkpt_path)

    print(f" Loading Model {i+1}: {arch} ({enc}) from {os.path.basename(chkpt_path)}...")
    try:
        model = get_model(architecture=arch, encoder=enc, validation=True)
        model = load_checkpoint_strict_without_aux(model, chkpt_path, device)

        ensemble_models.append(model)
    except Exception as e:
        print(f"  FATAL ERROR loading model {os.path.basename(chkpt_path)}: {e}")
        # In a final run, a failure to load a required model should be a fatal error.
        exit(1)

if len(ensemble_models) != n_models:
    print("\nError: Number of loaded models does not match the recipe. Exiting.")
    exit(1)

print(f"\nSuccessfully loaded all {len(ensemble_models)} models.")

In [ ]:
# # ### REFACTORED: Step 3 - Directly evaluate on the Test Set ###
# The val_loader and find_optimal_ensemble_threshold call are REMOVED.
print("\n--- Evaluating Ensemble Performance on Hold-Out Test Set ---")
ensemble_metrics = None
try:
  ensemble_metrics = analyze_ensemble_metrics(
      models_list=ensemble_models,
      constituent_models_info=constituent_models_info,
      model_weights=optimal_weights,
      test_loader=test_loader,
      device=device,
      optimal_threshold=optimal_threshold,
      train_mean=train_mean,
      train_std=train_std
  )
except Exception as e:
    print(f"Error during final ensemble evaluation: {e}")
    import traceback; traceback.print_exc()

In [ ]:
if ensemble_metrics is None:
  print("\nEnsemble evaluation failed. Exiting.")
  exit(1)
else:
  cm_out_png = os.path.join(os.path.dirname(ENSEMBLE_META_PATH), "confusion_matrix.png")
  tp = ensemble_metrics.get('confusion_matrix', {}).get('tp', 0)
  fp = ensemble_metrics.get('confusion_matrix', {}).get('fp', 0)
  fn = ensemble_metrics.get('confusion_matrix', {}).get('fn', 0)
  tn = ensemble_metrics.get('confusion_matrix', {}).get('tn', 0)
  save_confusion_matrix_png(tp, fp, fn, tn, cm_out_png)

In [ ]:
# --- Visualization & Reporting (Adjusted for final return structure) ---
if ensemble_metrics:
  visualize_ensemble_predictions(
      models_list=ensemble_models,
      model_weights=optimal_weights,
      dataloader=test_loader,
      device=device,
      threshold=optimal_threshold,
      num_samples=5,
      train_mean=train_mean,
      train_std=train_std,
      constituent_models_info=constituent_models_info,   # IMPORTANT for MajorityVote
      ensemble_method=ENSEMBLE_METHOD,                   # uses your global setting
      vote_thr=0.5,                                      # weighted majority
      show_prob_heatmap=True
  )

  print("\n" + "="*20 + " Final Summary From Returned Object " + "="*20)

  # Extract the main results dictionaries
  micro_results = ensemble_metrics.get('micro_averaged_metrics')
  macro_results = ensemble_metrics.get('macro_averaged_metrics')
  auc_score = ensemble_metrics.get('auc')

  if not micro_results or not macro_results:
      print("Evaluation produced no valid metrics.")
  else:
      # --- Print the Micro-Averages with their CIs ---
      print("\n--- Overall Pixel-Level Metrics (Micro-Averages) ---")
      micro_pes = micro_results.get('point_estimate', {})
      micro_cis = micro_results.get('ci', {})

      for metric in sorted(micro_pes.keys()):
          pe = micro_pes[metric]
          ci = micro_cis.get(metric, [0.0, 0.0])
          print(f"  {metric.replace('_',' ').title():<12}: {pe:.4f}  (95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])")

      # Report the AUC score
      if auc_score is not None:
          print(f"  {'AUC':<12}: {auc_score:.4f}  (CI not calculated via bootstrap)")


      # --- Print the Macro-Averages with their CIs ---
      print("\n--- Mean Per-Image Metrics (Macro-Averages) ---")
      macro_pes = macro_results.get('point_estimate', {})
      macro_cis = macro_results.get('ci', {})

      for metric in sorted(macro_pes.keys()):
          pe = macro_pes[metric]
          ci = macro_cis.get(metric, [0.0, 0.0])
          print(f"  {metric.replace('_',' ').title():<12}: {pe:.4f}  (95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])")

else:
    print("\nEnsemble evaluation failed.")

# --- ADD THE CUSTOMIZED REPORT CALL HERE ---
if ensemble_metrics:
    print_scientific_analysis_report(ensemble_metrics)
    # --- NEW: Call the CSV export function ---
    export_results_to_csv(
        ensemble_recipe=ensemble_recipe,
        metrics_results=ensemble_metrics,
        output_dir=os.path.dirname(ENSEMBLE_META_PATH) # Save CSV in the same folder as the recipe
    )

print("\n--- Final Evaluation Script Finished ---")

In [ ]:
# 2) Write + compile LaTeX report
tex_path, pdf_path = write_ensemble_report_latex(
    ensemble_recipe=ensemble_recipe,
    ensemble_metrics=ensemble_metrics,
    train_mean=train_mean,
    train_std=train_std,
    stats_sample_size=0,  # <- put the same value you used in dataset stats
    cm_png_path=cm_out_png,
    output_dir=os.path.dirname(ENSEMBLE_META_PATH),
)
print("TeX:", tex_path)
print("PDF:", pdf_path)